![Banner](banner.jpg)

# Laboratorio 7: Perceptrón Multicapa usando PyTorch

## 1. Introducción a Pytorch

Pytorch es una librería de alto rendimiento diseñada para calcular derivadas de forma automática, lo cual es esencial para el entrenamiento de modelos de **Deep Learning**. Adicionalmente, esta librería tiene una interfaz simple y conveniente para realizar los cálculos en la GPU, lo que permite acelerar los cómputos de manera considerable. En este tutorial, se van a presentar algunos de los conceptos más importantes para utilizar Pytorch en el entrenamiento de redes neuronales.

En Pytorch, la principal estructura de datos son los [**tensores**](https://pytorch.org/docs/stable/tensors.html), los cuales son muy similares a las matrices y arreglos de **NumPy**. A continuación, se presentan algunas de las formas más relevantes para crear tensores:

In [ ]:
!pip install torch numpy

In [ ]:


import torch

# Creación de un tensor
print('Crear un simple tensor')
tensor = torch.tensor([1])
print('torch.tensor([1]) =', tensor)
print('')

print('Crear un tensor en ceros')
tensor = torch.zeros((2, 2)) # Inicializar un tensor con 2 filas y dos columnas en cero
print('torch.zeros((2, 2)) =\n', tensor)
print('')

print('Crear un tensor con unos')
tensor = torch.ones((2, 2)) # Inicializar un tensor con 2 filas y dos columnas en uno
print('torch.zeros((2, 2)) =\n', tensor)
print('')

print('Crear un tensor con dos dimensiones')
tensor = torch.tensor([[1, 2, 3], [3, 2, 1]])
print(tensor)
print('')

print('Ver la forma del tensor')
print('tensor.shape = ', tensor.shape) # Dos filas, 3 columnas
print('')

print('Tensor con valores (enteros) aleatorios entre 0  y 5')
tensor = torch.randint(0, 6, (4,))
print(tensor)
print('')

print('Tensor con valores aleatorios generados a partir de una distribución normal con media 0 y varianza 1')
tensor = torch.randn((3,2))
print(tensor)
print('')

Algunos métodos de los tensores:

In [ ]:
print('Transformar a Numpy')
tensor = torch.randn((3,2)).numpy()
print(tensor)
print('')

print('Sumar los valores')
tensor = torch.ones((3,2)).sum()
print(tensor)
print('')

print('Sumar los valores por fila')
tensor = torch.ones((3,2)).sum(1)
print(tensor)
print('')

print('Promedio por columna')
tensor = torch.randn((3,2)).mean(0)
print(tensor)
print('')

print('Ver el tensor de otro manera. Verlo como vector')
tensor = torch.randn((2,2)).view(-1) # lo vismo que .view(4) porque el -1 representa el número necesario para encajar en una sola dimensión
print(tensor)
print('')

print('Transponer un tensor')
tensor = torch.randn((3,2)).t()
print(tensor)
print('')

print('Multiplicación de matrices')
tensor = torch.randn((2,3)) @ torch.randn((3,2)) # [2, 3] @ [3, 2] = [2, 2]
print(tensor)
print('')

print('Concatenar 2 tensores en la última dimensión')
tensor = torch.cat((torch.ones((2,2)), torch.randn((2,2))), dim=1) # tensor.shape = [2,4]
print(tensor)
print('')

print('Índices funcionan similar a numpy')
tensor = torch.ones((2, 2))[:,0] # Todas las filas solo la primera columna
print(tensor)
print('')

Una de las principales razones para utilizar Pytorch para implementar redes neuronales, es su capacidad de auto diferenciación, es decir, su capacidad de calcular las derivadas de manera automática, lo que permite el uso de diversas técnicas de optimización basadas en el gradiente para el entrenamiento de estas redes. A continuación se presenta un ejemplo de cómo utilizar este mecanismo:

In [ ]:
# Si tenemos 2x + 3x + x = y, la derivada de y con respecto a x será de 2+3+1=6
x = torch.randn((1,), requires_grad=True) # x es un número cualquiera, y se debe indicar que se va a calcular la derivada con respecto a x mediante requires_grad=True
y_  = torch.tensor([2, 3, 1]) * x # y_ = [2, 3, 1] * x, donde por broadcasting (similar a numpy https://pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html#in-brief-tensor-broadcasting)
# y_ = [2, 3, 1] * x =  [2, 3, 1] * [x, x, x] = [2x, 3x, 1x]
y = y_.sum() # y = 2x + 3x + x
print('Valor de y:', y)

# Pedir a Pytorch que calcule la derivada de y con respecto a las variables que tienen requires_grad=True
y.backward()

# Derivada de y con respecto a x
print('dy/dx =', x.grad)

Puede ver más información sobre la diferenciación automática en Pytorch [aquí](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html). Adicionalmente, se recomienda que realice el [tutorial básico de Pytorch](https://pytorch.org/tutorials/beginner/basics/intro.html).

## 2. Identificación de dígitos con MLP

En este laboratorio vamos a explorar cómo construir un **Perceptrón Multicapa (MLP)** utilizando **PyTorch** para resolver una de las tareas clásicas del aprendizaje automático: la **identificación de dígitos escritos a mano**.

El dataset que utilizaremos es MNIST, que se ha convertido en el estándar por excelencia para enseñar y evaluar modelos de clasificación de imágenes. MNIST contiene **70,000 imágenes** de dígitos del 0 al 9, escritos a mano, en escala de grises, con una resolución de **28x28 píxeles**.

### ¿Qué es un Perceptrón Multicapa (MLP)?

Un MLP es un tipo de red neuronal artificial compuesta por **múltiples capas de neuronas** completamente conectadas:

- **Capa de entrada**: Recibe los datos (en nuestro caso, los 784 píxeles de cada imagen).
- **Capas ocultas**: Capas intermedias que aprenden representaciones cada vez más abstractas de los datos.
- **Capa de salida**: Produce la predicción final (en nuestro caso, las probabilidades de cada uno de los 10 dígitos).

Cada neurona aplica una **transformación lineal** seguida de una **función de activación no lineal** (como ReLU), lo que permite al modelo aprender relaciones complejas en los datos.

### Objetivos del laboratorio

1. Cargar y explorar el dataset MNIST.
2. Preprocesar las imágenes para que puedan ser utilizadas por una red neuronal.
3. Construir un MLP desde cero usando PyTorch.
4. Entrenar el modelo y evaluar su desempeño.
5. Visualizar los resultados y las predicciones del modelo.

## 3. Preparación del entorno

### 3.1. Instalación de paquetes

Primero, instalemos las librerías necesarias para este laboratorio.

In [ ]:
!pip install torch torchvision matplotlib numpy

### 3.2. Importación de librerías

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo utilizado: {device}')

Versiones utilizadas de las librerías importadas:

In [ ]:
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)

## 4. Carga y exploración del dataset MNIST

El dataset MNIST se puede descargar directamente mediante `torchvision.datasets`. Este dataset ya viene dividido en un conjunto de **entrenamiento** (60,000 imágenes) y un conjunto de **prueba** (10,000 imágenes).

Para preparar los datos, aplicaremos dos transformaciones:
1. `ToTensor()`: Convierte las imágenes PIL a tensores de PyTorch con valores entre 0 y 1.
2. `Normalize((0.1307,), (0.3081,))`: Normaliza los valores con la media y desviación estándar del dataset MNIST.

In [ ]:
# Definir las transformaciones para el preprocesamiento
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Descargar y cargar el dataset de entrenamiento
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Descargar y cargar el dataset de prueba
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f'Tamaño del conjunto de entrenamiento: {len(train_dataset)}')
print(f'Tamaño del conjunto de prueba: {len(test_dataset)}')
print(f'Forma de una imagen: {train_dataset[0][0].shape}')
print(f'Número de clases: {len(train_dataset.classes)}')

### 4.1. Visualización de ejemplos del dataset

Antes de construir nuestro modelo, es importante familiarizarnos con los datos. Visualicemos algunas de las imágenes del dataset junto con sus etiquetas correspondientes.

In [ ]:
# Visualizar una cuadrícula de imágenes de ejemplo
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Ejemplos del dataset MNIST', fontsize=14)

for i, ax in enumerate(axes.flatten()):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Dígito: {label}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

### 4.2. Exploración de la distribución de clases

Es buena práctica verificar que el dataset esté balanceado, es decir, que cada dígito tenga una cantidad similar de ejemplos.

In [ ]:
# Contar la cantidad de ejemplos por clase
labels = [label for _, label in train_dataset]
unique, counts = np.unique(labels, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(unique, counts, color='steelblue')
plt.xlabel('Dígito')
plt.ylabel('Cantidad de ejemplos')
plt.title('Distribución de clases en el conjunto de entrenamiento')
plt.xticks(unique)
plt.show()

for digit, count in zip(unique, counts):
    print(f'Dígito {digit}: {count} ejemplos')

### 4.3. Creación de DataLoaders

Para entrenar nuestro modelo de forma eficiente, vamos a utilizar `DataLoader` de PyTorch, que nos permite cargar los datos en **lotes (batches)**. Esto es esencial ya que:

- Permite al modelo ver varios ejemplos al tiempo, proporcionando una señal de aprendizaje más estable.
- Mejora la eficiencia computacional al aprovechar el paralelismo.
- Facilita el uso de gradiente descendiente estocástico (SGD) por mini-lotes.

In [ ]:
# Tamaño del lote
batch_size = 64

# Crear los DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Verificar la forma de un lote
images, labels = next(iter(train_loader))
print(f'Forma del lote de imágenes: {images.shape}')
print(f'Forma del lote de etiquetas: {labels.shape}')
print(f'Esto significa que tenemos {images.shape[0]} imágenes de {images.shape[2]}x{images.shape[3]} píxeles')

## 5. Construcción del modelo MLP

Ahora que tenemos los datos listos, vamos a construir nuestro Perceptrón Multicapa. La arquitectura de nuestro modelo será la siguiente:

1. **Capa de entrada**: 784 neuronas (28 × 28 píxeles aplanados).
2. **Primera capa oculta**: 256 neuronas con activación ReLU.
3. **Segunda capa oculta**: 128 neuronas con activación ReLU.
4. **Capa de salida**: 10 neuronas (una por cada dígito del 0 al 9).

Recordemos que cada neurona aplica la operación: $\mathbf{y} = f(\mathbf{W} \cdot \mathbf{x} + \mathbf{b})$

Donde $\mathbf{W}$ son los pesos, $\mathbf{b}$ es el sesgo, y $f$ es la función de activación.

### 5.1. Implementación manual (desde cero)

Primero, implementemos el MLP de forma manual para comprender cómo funcionan las capas por dentro. Definiremos los pesos y sesgos como tensores de PyTorch con `requires_grad=True` para que el mecanismo de autodiferenciación pueda calcular las derivadas automáticamente.

In [ ]:
# Dimensiones del modelo
input_size = 28 * 28   # 784 píxeles
hidden1_size = 256     # Primera capa oculta
hidden2_size = 128     # Segunda capa oculta
output_size = 10       # 10 dígitos (0-9)

# Parámetros de la primera capa oculta
W1 = (torch.randn(input_size, hidden1_size, device=device) * 0.01).requires_grad_(True)
b1 = torch.zeros(hidden1_size, device=device, requires_grad=True)

# Parámetros de la segunda capa oculta
W2 = (torch.randn(hidden1_size, hidden2_size, device=device) * 0.01).requires_grad_(True)
b2 = torch.zeros(hidden2_size, device=device, requires_grad=True)

# Parámetros de la capa de salida
W3 = (torch.randn(hidden2_size, output_size, device=device) * 0.01).requires_grad_(True)
b3 = torch.zeros(output_size, device=device, requires_grad=True)

parameters = [W1, b1, W2, b2, W3, b3]
total_params = sum(p.numel() for p in parameters)
print(f'Número total de parámetros: {total_params:,}')
print(f'  - Capa 1: W1 {W1.shape}, b1 {b1.shape} = {W1.numel() + b1.numel():,} parámetros')
print(f'  - Capa 2: W2 {W2.shape}, b2 {b2.shape} = {W2.numel() + b2.numel():,} parámetros')
print(f'  - Capa 3: W3 {W3.shape}, b3 {b3.shape} = {W3.numel() + b3.numel():,} parámetros')

In [ ]:
def mlp_manual(x):
    """MLP implementado manualmente."""
    # Aplanar la imagen de 28x28 a un vector de 784
    x = x.view(-1, 28 * 28)

    # Primera capa oculta: transformación lineal + ReLU
    h1 = F.relu(x @ W1 + b1)

    # Segunda capa oculta: transformación lineal + ReLU
    h2 = F.relu(h1 @ W2 + b2)

    # Capa de salida: transformación lineal (sin activación, ya que usaremos CrossEntropyLoss)
    logits = h2 @ W3 + b3

    return logits

# Verificar que el modelo funcione correctamente
sample_batch = images.to(device)
output = mlp_manual(sample_batch)
print(f'Forma de la entrada: {sample_batch.shape}')
print(f'Forma de la salida: {output.shape}')
print(f'\nLos logits para la primera imagen del lote son:')
print(output[0])

### 5.2. Entrenamiento del modelo manual

Vamos a entrenar nuestro modelo manual utilizando:
- **Función de pérdida**: `CrossEntropyLoss`, que combina LogSoftmax y NLLLoss. Es la función estándar para problemas de clasificación multiclase.
- **Optimizador**: Gradiente descendiente estocástico (SGD) con una tasa de aprendizaje fija.

In [ ]:
lr = 0.01  # Tasa de aprendizaje
epochs = 5 # Número de épocas

losses = []

for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        # Forward: calcular las predicciones
        logits = mlp_manual(batch_images)

        # Calcular la pérdida (Cross Entropy Loss)
        loss = F.cross_entropy(logits, batch_labels)

        # Backward: calcular los gradientes
        loss.backward()

        # Actualizar los parámetros manualmente (SGD)
        with torch.no_grad():
            for p in parameters:
                p.data -= lr * p.grad
                p.grad.zero_()

        # Registrar métricas
        epoch_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == batch_labels).sum().item()
        total += batch_labels.size(0)

    avg_loss = epoch_loss / len(train_loader)
    accuracy = correct / total
    losses.append(avg_loss)
    print(f'Época {epoch+1}/{epochs} - Pérdida: {avg_loss:.4f} - Precisión: {accuracy:.4f}')

In [ ]:
# Visualizar la curva de pérdida durante el entrenamiento
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), losses, marker='o', color='steelblue')
plt.xlabel('Época')
plt.ylabel('Pérdida (Loss)')
plt.title('Curva de aprendizaje - Modelo Manual')
plt.grid(True, alpha=0.3)
plt.show()

### 5.3. Evaluación del modelo manual en el conjunto de prueba

Evaluemos el desempeño de nuestro modelo sobre datos que nunca ha visto (el conjunto de prueba).

In [ ]:
@torch.no_grad()
def evaluate_manual(model_fn, data_loader):
    """Evalúa un modelo en un conjunto de datos."""
    correct = 0
    total = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model_fn(images)
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return accuracy

accuracy = evaluate_manual(mlp_manual, test_loader)
print(f'Precisión del modelo manual en el conjunto de prueba: {accuracy:.4f} ({accuracy*100:.2f}%)')

## 6. Implementación con `nn.Module` de PyTorch

Ahora que hemos comprendido cómo funciona un MLP por dentro, vamos a utilizar las herramientas de alto nivel de PyTorch para construir el mismo modelo de una forma más elegante y mantenible.

La clase `nn.Module` de PyTorch nos permite definir modelos de forma modular, encapsulando los parámetros y la lógica del modelo en una sola clase.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super(MLP, self).__init__()
        # Definir las capas del modelo
        self.fc1 = nn.Linear(input_size, hidden1_size)   # Capa 1: entrada -> oculta 1
        self.fc2 = nn.Linear(hidden1_size, hidden2_size) # Capa 2: oculta 1 -> oculta 2
        self.fc3 = nn.Linear(hidden2_size, output_size)  # Capa 3: oculta 2 -> salida

    def forward(self, x):
        # Aplanar la imagen de 28x28 a un vector de 784
        x = x.view(-1, 28 * 28)

        # Pasar por las capas con activación ReLU
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

# Crear una instancia del modelo
model = MLP(input_size=784, hidden1_size=256, hidden2_size=128, output_size=10).to(device)
print(model)
print(f'\nNúmero total de parámetros: {sum(p.numel() for p in model.parameters()):,}')

### 6.1. Definición de la función de pérdida y el optimizador

Para entrenar nuestro modelo, necesitamos definir:
- **Función de pérdida**: `CrossEntropyLoss`, que internamente aplica softmax a los logits y calcula la entropía cruzada con las etiquetas reales.
- **Optimizador**: Usaremos **Adam**, un optimizador más avanzado que SGD, que adapta la tasa de aprendizaje de cada parámetro de forma individual.

In [ ]:
# Función de pérdida
criterion = nn.CrossEntropyLoss()

# Optimizador Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 6.2. Funciones de entrenamiento y evaluación

Definamos funciones reutilizables para el ciclo de entrenamiento y la evaluación del modelo.

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer):
    """Entrena el modelo por una época completa."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward: calcular predicciones
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward: calcular gradientes
        optimizer.zero_grad()
        loss.backward()

        # Actualizar parámetros
        optimizer.step()

        # Registrar métricas
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(train_loader), correct / total


@torch.no_grad()
def evaluate(model, data_loader, criterion):
    """Evalúa el modelo en un conjunto de datos."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(data_loader), correct / total

### 6.3. Entrenamiento del modelo

Ahora, entrenemos nuestro modelo por varias épocas y registremos tanto la pérdida como la precisión en entrenamiento y prueba.

In [ ]:
epochs = 10

# Listas para almacenar las métricas
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print('Entrenamiento del modelo MLP con nn.Module')
print('=' * 60)

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f'Época {epoch+1:2d}/{epochs} | '
          f'Pérdida train: {train_loss:.4f} | Precisión train: {train_acc:.4f} | '
          f'Pérdida test: {test_loss:.4f} | Precisión test: {test_acc:.4f}')

print('=' * 60)
print(f'Precisión final en test: {test_accuracies[-1]*100:.2f}%')

### 6.4. Visualización de las métricas de entrenamiento

Visualicemos las curvas de pérdida y precisión durante el entrenamiento para verificar que el modelo esté aprendiendo correctamente.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Curva de pérdida
ax1.plot(range(1, epochs + 1), train_losses, marker='o', label='Entrenamiento', color='steelblue')
ax1.plot(range(1, epochs + 1), test_losses, marker='s', label='Prueba', color='coral')
ax1.set_xlabel('Época')
ax1.set_ylabel('Pérdida')
ax1.set_title('Curva de pérdida')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Curva de precisión
ax2.plot(range(1, epochs + 1), train_accuracies, marker='o', label='Entrenamiento', color='steelblue')
ax2.plot(range(1, epochs + 1), test_accuracies, marker='s', label='Prueba', color='coral')
ax2.set_xlabel('Época')
ax2.set_ylabel('Precisión')
ax2.set_title('Curva de precisión')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Análisis de resultados

### 7.1. Visualización de predicciones

Veamos algunas predicciones del modelo sobre imágenes del conjunto de prueba. Mostraremos tanto las predicciones correctas como las incorrectas.

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, num_images=10):
    """Muestra predicciones del modelo sobre imágenes aleatorias."""
    model.eval()
    indices = np.random.choice(len(dataset), num_images, replace=False)

    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    fig.suptitle('Predicciones del modelo MLP', fontsize=14)

    for i, ax in enumerate(axes.flatten()):
        image, true_label = dataset[indices[i]]
        image_device = image.unsqueeze(0).to(device)
        logits = model(image_device)
        probs = F.softmax(logits, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred_label].item()

        ax.imshow(image.squeeze(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'Pred: {pred_label} ({confidence:.1%})\nReal: {true_label}',
                     color=color, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset)

### 7.2. Matriz de confusión

La matriz de confusión nos permite observar qué dígitos son más difíciles de clasificar para el modelo, y con cuáles dígitos tiende a confundirlos.

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, data_loader, num_classes=10):
    """Calcula la matriz de confusión."""
    model.eval()
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    for images, labels in data_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        predicted = predicted.cpu()

        for t, p in zip(labels, predicted):
            confusion[t, p] += 1

    return confusion

confusion = compute_confusion_matrix(model, test_loader)

plt.figure(figsize=(10, 8))
plt.imshow(confusion, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de confusión')
plt.xticks(range(10))
plt.yticks(range(10))

# Agregar los valores numéricos dentro de cada celda
for i in range(10):
    for j in range(10):
        color = 'white' if confusion[i, j] > confusion.max() / 2 else 'black'
        plt.text(j, i, str(confusion[i, j].item()),
                 ha='center', va='center', color=color, fontsize=9)

plt.tight_layout()
plt.show()

### 7.3. Precisión por dígito

Podemos observar qué dígitos son más fáciles y cuáles más difíciles de clasificar para nuestro modelo.

In [ ]:
# Calcular la precisión por dígito
per_class_accuracy = confusion.diag().float() / confusion.sum(dim=1).float()

plt.figure(figsize=(8, 4))
colors = ['green' if acc > 0.97 else 'orange' if acc > 0.95 else 'red' for acc in per_class_accuracy]
plt.bar(range(10), per_class_accuracy.numpy(), color=colors)
plt.xlabel('Dígito')
plt.ylabel('Precisión')
plt.title('Precisión por dígito')
plt.xticks(range(10))
plt.ylim(0.9, 1.0)
plt.grid(True, alpha=0.3, axis='y')

for i, acc in enumerate(per_class_accuracy):
    plt.text(i, acc + 0.002, f'{acc:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Experimentación: Efecto de la arquitectura

Una parte importante del aprendizaje profundo es experimentar con diferentes arquitecturas y observar cómo afectan el desempeño del modelo. Comparemos diferentes configuraciones del MLP.

In [ ]:
# Definir diferentes arquitecturas para comparar
architectures = {
    'Pequeño (64)': [64],
    'Mediano (256, 128)': [256, 128],
    'Grande (512, 256, 128)': [512, 256, 128],
}

class FlexibleMLP(nn.Module):
    """MLP con número variable de capas ocultas."""
    def __init__(self, input_size, hidden_sizes, output_size):
        super(FlexibleMLP, self).__init__()
        layers = []
        prev_size = input_size
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            layers.append(nn.ReLU())
            prev_size = h_size
        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        return self.network(x)

results = {}

for name, hidden_sizes in architectures.items():
    print(f'\nEntrenando arquitectura: {name}')
    m = FlexibleMLP(784, hidden_sizes, 10).to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  Parámetros: {n_params:,}')

    best_acc = 0
    for epoch in range(5):
        train_epoch(m, train_loader, criterion, opt)
        _, test_acc = evaluate(m, test_loader, criterion)
        best_acc = max(best_acc, test_acc)
        print(f'  Época {epoch+1}/5 - Precisión test: {test_acc:.4f}')

    results[name] = {'accuracy': best_acc, 'params': n_params}

print('\n' + '=' * 50)
print('Resumen de resultados:')
print('=' * 50)
for name, res in results.items():
    print(f'{name:30s} | Precisión: {res["accuracy"]*100:.2f}% | Parámetros: {res["params"]:>10,}')

## 9. Conclusiones

En este laboratorio hemos explorado cómo construir un **Perceptrón Multicapa (MLP)** utilizando PyTorch para la tarea de **identificación de dígitos manuscritos** con el dataset MNIST. A continuación, resumimos los principales aprendizajes:

1. **El MLP es capaz de alcanzar un alto desempeño** (~97-98%) en la tarea de clasificación de dígitos, lo que demuestra la capacidad de las redes neuronales para aprender patrones complejos en los datos.

2. **La implementación manual vs. `nn.Module`**: Implementar el modelo desde cero nos ayuda a comprender los mecanismos internos (pesos, sesgos, gradientes), mientras que `nn.Module` nos proporciona una interfaz limpia y eficiente para trabajar en proyectos más complejos.

3. **El preprocesamiento es fundamental**: La normalización de los datos y la correcta estructuración en batches son pasos esenciales para el entrenamiento eficiente del modelo.

5. **Limitaciones del MLP**: A pesar de los buenos resultados, los MLPs tienen limitaciones para el procesamiento de imágenes, ya que no consideran la estructura espacial de los píxeles. Para tareas más avanzadas de visión por computadora, se utilizan **Redes Neuronales Convolucionales (CNNs)**, que aprenden a detectar patrones locales como bordes y texturas.